In [ ]:
%%configure -f
{
    "conf": {
        "spark.dynamicAllocation.enabled": "false",
        "spark.driver.cores": "4",
        "spark.driver.memory": "28g",
        "spark.executor.cores": "4",
        "spark.executor.memory": "28g",
        "spark.executor.instances": 1
    }
}

# Data Preparation

**Scenario:** supermarket_net_sales_forecast  
**Generated:** 2026-06-03  
**Key Parameters:**
- Forecast Horizon: TBD (set in NB06)
- Time Granularity: Weekly (Thursdays)
- Target Column: TOTAL_NET_SALES
- Unique ID: STORE_LOCATION_ID
- Input Table: ts_mmm.df_raw
- Output Table: ts_mmm.supermarket_net_sales_forecast_prepared

This series of notebooks helps data scientists to forecast multiple time series by building models based on the time-series profiling, i.e identifying similar consumption profiles and building a specific model for each cluster.

Clustering profile of time series data helps in defining the best fitting model by understanding in terms of choice of regressors (calendar variables or temperatures), forecasting algorithm (ARIMA vs Exponential smoothing) and train set (one year or just few days of data).

The steps to clustering time series are the following:
1. Prepare data by understand the structure of the data (cross-sectional vs panel data), build a full time sequence and treating missing values
2. Identifying intermittent time series vs smooth time series
3. Clustering with K-Means those time-series identified as **smooth**

In this notebook you will follow step 1 on data preparation.

# Summary of tasks
This notebook is to prepare data for time-series profiling.

As a **first step**, you need to understand the type of data you are dealing with. You might have **panel data** or **cross-sectional data**.

The output of this first step is to:
1. Identify if data are panel or cross-section
2. If panel data, what is the unique-id variable
3. Identify the granularity or frequency of the time series
4. Set the time series variable as a datetime object

As a **second step**, you need to build a full time sequence, merge the data on it to understand if there are missing values in the full time sequence and count the number of NAs

**Finally**, check if there are no duplicated entries.

# Implementation

## Packages

In [ ]:
# Data elaboration functions
import pandas as pd
import string
import numpy as np
import re
from typing import List, Any, Dict, Union
from pathlib import Path

# Datetime functions
import datetime as dt

# Plot functions
import matplotlib.pyplot as plt
%matplotlib inline

# Spark for Fabric
from pyspark.sql import SparkSession

print("\u2705 Packages loaded successfully")

## Inlined Utility Functions
These functions are inlined from the original `Utils` module for Fabric compatibility

In [ ]:
# ============================================================
# INLINED UTILITY FUNCTIONS (from Utils.utils)
# ============================================================

def camel_to_snake(name: str) -> str:
    """
    Changes string from camel case to snake case
    """
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()


def columns_camel_to_snake(df: pd.DataFrame) -> pd.DataFrame:
    """
    Changes dataframe columns from camel case to snake case
    """
    list_cols = list(df.columns)
    for name in list_cols:
        new_name = camel_to_snake(name)
        df.rename(columns={name: new_name}, inplace=True)
    return df


def add_seq(df: pd.DataFrame, date_var: str, serie: str, freq: str, end_date: str = '', start_date: str = '') -> pd.DataFrame:
    """
    Creates a sequence of complete dates to a dataframe.
    CUSTOMIZED: Uses extended end_date to handle freq='W' (W-SUN) boundary issue with Thursday data.
    """
    if hasattr(df[date_var].dtype, 'tz') and df[date_var].dtype.tz is not None:
        df.loc[:, date_var] = df[date_var].apply(lambda x: x.tz_localize(None) if hasattr(x, 'tz_localize') else x)

    period_freq_mapping = {'MS': 'M', 'M': 'M', 'D': 'D', 'H': 'H', 'W': 'W', 'W-SUN': 'W', 'Y': 'Y', 'YS': 'Y', 'Q': 'Q', 'QS': 'Q'}
    period_freq = period_freq_mapping.get(freq, freq)

    seq = pd.DataFrame()
    serie_list = list(df.loc[:, serie].unique())

    for i in serie_list:
        if start_date == '':
            s_date = min(df.loc[df[serie] == i, date_var])
            if hasattr(s_date, 'tz_localize'):
                try:
                    s_date = s_date.tz_localize(None)
                except:
                    pass
        else:
            s_date = pd.to_datetime(start_date, dayfirst=True)
        
        if end_date == '':
            e_date = max(df.loc[df[serie] == i, date_var])
            if hasattr(e_date, 'tz_localize'):
                try:
                    e_date = e_date.tz_localize(None)
                except:
                    pass
        else:
            e_date = pd.to_datetime(end_date, dayfirst=True)
        
        time_range = pd.Series(pd.date_range(start=s_date, end=e_date, freq=freq)).dt.to_period(period_freq)
        
        print('Adding sequence to serie', i, 'as', serie_list.index(i) + 1, 'of', len(serie_list))
        temp = pd.DataFrame.from_dict({serie: [i] * len(time_range), 'date': time_range})
        temp.rename(columns={'date': date_var}, inplace=True)
        seq = pd.concat([seq, temp], axis=0, ignore_index=True)
    
    duplicates = seq.loc[:, [serie, date_var]].duplicated().any()
    if duplicates:
        raise Exception("add_seq: there are duplicates in sequence")
    else:
        print("add_seq: there are NO duplicates in sequence")
    
    print('Total series to forecast:', len(seq.loc[:, [serie, date_var]].drop_duplicates()))
    
    return seq


def check_length_time_serie(df: pd.DataFrame, date_var: str, index: str, freq: Union[str, None] = None) -> pd.DataFrame:
    """
    Checks the length that a time sequence should have.
    NOTE: May report 'NOT OK' for weekly Thursday data due to freq='W' (Sunday-anchored) boundary mismatch.
    This is a known diagnostic artifact - verify actual count matches expectations.
    """
    pivot = pd.pivot_table(df, index=index, values=date_var, aggfunc=['count', 'min', 'max']).reset_index()
    pivot.columns = pivot.columns.get_level_values(0)
    pivot.loc[:, 'count'] = pivot.loc[:, 'count'].astype(float)
    
    freq = freq or globals().get('frequency')
    if freq is None:
        print('check_length_time_serie: no frequency provided; cannot compute expected_obs')
        pivot.loc[:, 'expected_obs'] = np.nan
        return pivot[[index, 'count', 'expected_obs']].drop_duplicates()
    
    freq_mapping = {
        'M': 'MS', 'MS': 'MS',
        'D': 'D',
        'H': 'H',
        'W': 'W', 'W-SUN': 'W-SUN',
        'Y': 'YS', 'YS': 'YS',
        'Q': 'QS', 'QS': 'QS',
    }
    dr_freq = freq_mapping.get(freq, freq)
    pivot.loc[:, 'freq'] = freq
    pivot.loc[:, 'expected_obs'] = pivot.apply(
        lambda row: len(pd.date_range(start=row['min'], end=row['max'], freq=dr_freq)), axis=1
    )
    pivot.loc[:, 'mismatch'] = (pivot['count'] != pivot['expected_obs']).astype(int)
    
    if sum(pivot.mismatch) > 0:
        print('Expected length of sequence is NOT OK')
    else:
        print('Expected length of sequence is OK')
    
    return pivot[[index, 'count', 'expected_obs']].drop_duplicates()


print("\u2705 Utility functions loaded")

## Configuration Parameters
### \u2705 CHECK POINT with the data scientist: confirm configuration parameters
Confirm the date column, target column, unique_id definition, and frequency before proceeding.

In [ ]:
# ============================================================
# CONFIGURATION PARAMETERS
# CUSTOMIZED: Lakehouse/table names for supermarket scenario
# ============================================================

# Time series configuration
date_var = 'WEEK_START_DT'
date_format = '%Y-%m-%d'
id = 'STORE_LOCATION_ID'
unique_id = 'STORE_LOCATION_ID'
list_unique_id = ['STORE_LOCATION_ID', 'REGION']
frequency = 'W'  # Weekly frequency
y = 'TOTAL_NET_SALES'

# Fabric Lakehouse configuration
# CUSTOMIZED: Using ts_mmm lakehouse and scenario-specific output table
LAKEHOUSE_NAME = "ts_mmm"
INPUT_TABLE = "df_raw"
OUTPUT_TABLE = "supermarket_net_sales_forecast_prepared"

print("\ud83d\udccb Configuration loaded:")
print(f"  - Date variable: {date_var}")
print(f"  - Unique ID: {unique_id}")
print(f"  - Target variable: {y}")
print(f"  - Frequency: {frequency}")
print(f"  - Input table: {LAKEHOUSE_NAME}.{INPUT_TABLE}")
print(f"  - Output table: {LAKEHOUSE_NAME}.{OUTPUT_TABLE}")

## Load Data
Load data from Lakehouse table

In [ ]:
# ============================================================
# LOAD DATA FROM LAKEHOUSE
# CUSTOMIZED: Using just table name (session attached to lakehouse)
# ============================================================

try:
    df_spark = spark.table(f"{INPUT_TABLE}")
    df = df_spark.toPandas()
except Exception as e:
    print(f"\u274c Error loading data: {e}")
    raise

print(f"\u2705 Loaded {len(df)} rows, {len(df.columns)} columns")
print(df.head())

### Defining and formatting the date variable

In [ ]:
# Convert date column to datetime
df.loc[:, 'date_converted'] = pd.to_datetime(df.loc[:, date_var], format=date_format)
df.drop(columns=[date_var], inplace=True)
df.rename(columns={'date_converted': date_var}, inplace=True)

# Reorder columns
df.ordered_columns = [unique_id, date_var, y] + [col for col in df.columns if col not in [unique_id, date_var, y]]
df = df.reindex(columns=df.ordered_columns)

# Verify day-of-week (expect Thursday = 3)
dow_values = df[date_var].dt.dayofweek.unique()
print(f"Day of week values: {dow_values} (3=Thursday)")
print(f"Date range: {df[date_var].min()} to {df[date_var].max()}")
df.head()

### \u2705 CHECK POINT with the data scientist: check missing values
Check number of columns and NaNs before generating full time sequences

In [ ]:
print("Df columns: ", list(df.columns))
print(f"Shape: {df.shape}")
print("NaNs:", df.isna().sum().values.sum())

# Data Preparation
Create a full time sequence on a chosen frequency and aggregate.

## Dealing with NAs by creating a full time sequence
### \u2705 CHECK POINT with the data scientist: list unique_id

In [ ]:
print('List ids:', list(df[unique_id].unique()))
print(f'Number of unique IDs: {len(list(df[unique_id].unique()))}')

## Creating a full time sequence
### \u2705 CHECK POINT with the data scientist: check the frequency of the time series

In [ ]:
print("Frequency is", frequency)

### Create a sequence of dates

In [ ]:
# CUSTOMIZED: Extended end_date to '30/03/2025' to handle freq='W' (W-SUN) boundary issue.
# The data runs Thu 2023-04-06 to Thu 2025-03-27. With freq='W' generating Sundays,
# the last Sunday before 2025-03-27 is 2025-03-23, missing the final week.
# Extending end_date past the next Sunday (2025-03-30) ensures all 104 periods are captured.

freq_mapping = {'M': 'MS', 'MS': 'MS', 'D': 'D', 'H': 'H', 'W': 'W', 'W-SUN': 'W-SUN', 'Q': 'QS', 'QS': 'QS', 'Y': 'YS', 'YS': 'YS'}
pandas_freq = freq_mapping.get(frequency, frequency)

df_seq = add_seq(df, date_var=date_var, serie=unique_id, freq=pandas_freq, end_date='30/03/2025', start_date='')
print(f"Sequence shape: {df_seq.shape}")

## Merge full time sequence with df
Use a left 1:1 merge on the df with the full time sequence

In [ ]:
df.loc[:, date_var] = df.loc[:, date_var].dt.to_period(frequency)
df.head()

In [ ]:
df_time_sequence = pd.merge(df_seq, df, on=[unique_id, date_var], how='left', validate='1:1')
print(f"Merged shape: {df_time_sequence.shape}")
print(f"NaNs after merge: {df_time_sequence[y].isna().sum()}")
df_time_sequence.head()

In [ ]:
# CUSTOMIZED: Convert period to timestamp and shift from Monday (period start) back to Thursday
# period.to_timestamp() returns the start of the period (Monday for W-SUN weeks).
# Adding 3 days shifts to Thursday to match the original data's day-of-week.
df_time_sequence.loc[:, date_var] = df_time_sequence.loc[:, date_var].apply(
    lambda x: x.to_timestamp() + pd.Timedelta(days=3)
)

# Verify Thursday restoration
dow_check = df_time_sequence[date_var].dt.dayofweek.unique()
print(f"Day of week after restoration: {dow_check} (3=Thursday)")
print(f"Date range: {df_time_sequence[date_var].min()} to {df_time_sequence[date_var].max()}")
df_time_sequence.head()

## Count the number of obs per time series
### \u2705 CHECK POINT with the data scientist: count observations per id
NOTE: `check_length_time_serie` may report 'NOT OK' due to freq='W' (Sunday-anchored) boundary mismatch with Thursday data. This is a diagnostic artifact - the actual count (104) is correct.

In [ ]:
length_check = check_length_time_serie(df_time_sequence, date_var, index=unique_id, freq=frequency)
print(length_check.head(5).to_string())
print(f"\nAll series have same count: {length_check['count'].nunique() == 1}")
print(f"Count per series: {int(length_check['count'].iloc[0])}")

### List ids after resampling

In [ ]:
print('List ids after resampling:', list(df_time_sequence[unique_id].unique()))

# Plots
Visualize sample time series

In [ ]:
# Simple matplotlib plot for a sample of time series
list_ids_to_plot = list(df_time_sequence[unique_id].unique())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

sample_ids = list_ids_to_plot[:4]

for idx, ts_id in enumerate(sample_ids):
    ts_data = df_time_sequence[df_time_sequence[unique_id] == ts_id].sort_values(date_var)
    axes[idx].plot(ts_data[date_var], ts_data[y], marker='o', markersize=3)
    axes[idx].set_title(f'Time Series: Store {ts_id}')
    axes[idx].set_xlabel('Date')
    axes[idx].set_ylabel(y)
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\u2705 Plotted {len(sample_ids)} sample time series")

# Count NAs in target by unique_id

In [ ]:
# Count NAs by unique_id
na_counts = df_time_sequence[df_time_sequence[y].isna()].groupby(unique_id).size().reset_index(name=f'{y}_count_NA')

if len(na_counts) > 0:
    print("NAs found by unique_id:")
    print(na_counts)
else:
    print("\u2705 No NAs found in target variable!")

# \u2705 CHECK POINT with the data scientist: always check for duplicate values
Remove duplicates: always check for the presence of duplicate values

In [ ]:
df_final = df_time_sequence.drop_duplicates()
print('List ids in df_final:', list(df_final[unique_id].unique()))
assert df_final[df_final.duplicated()].count().sum() == 0, "y should not contain duplicates"
print(f'\u2705 No duplicates found')
print(f'Min date: {df_final[date_var].min()}')
print(f'Max date: {df_final[date_var].max()}')
print(f'Shape: {df_final.shape}')

# Saving
Save the prepared dataframe to Lakehouse table

In [ ]:
# ============================================================
# SAVE TO LAKEHOUSE TABLE
# CUSTOMIZED: Using overwriteSchema option for schema evolution
# ============================================================

df_final_spark = spark.createDataFrame(df_final)
df_final_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OUTPUT_TABLE}")

print(f"\u2705 df_final saved to: {LAKEHOUSE_NAME}.{OUTPUT_TABLE}")
print(f"Rows: {len(df_final)}, Columns: {len(df_final.columns)}")
print(f'\ud83d\udd0d Min date: {df_final[date_var].min()}')
print(f'\ud83d\udd0d Max date: {df_final[date_var].max()}')
df_final.head()